# Modelagem do Perceptron

Este notebook explora a implementação de um Perceptron simples, uma versão em classe, a função sigmoide contínua e um exemplo de rede neural multicamadas (MLP). O objetivo é mostrar a evolução desde o modelo linear básico até uma arquitetura com múltiplas camadas e transformações de dados.


## Funções básicas do Perceptron

Neste bloco definimos as funções fundamentais para o funcionamento de um perceptron:

- `sigma(v)`: função de ativação degrau, que retorna `1` se `v >= 0` e `0` caso contrário.
- `predict(x, w, b)`: calcula a previsão do perceptron para uma entrada `x`, fazendo o produto escalar `w @ x + b` e aplicando a função de ativação.
- `train(X, y, w, b, eta, epochs)`: implementa o algoritmo de aprendizado do perceptron. Para cada época e para cada exemplo `(x, y_true)`, calcula `y_hat`, mede o erro `y_true - y_hat` e atualiza os pesos `w` e o bias `b` usando a taxa de aprendizado `eta`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def sigma(v):
    return 1 if v >= 0 else 0


def predict(x, w, b):
    return sigma(w @ x + b)


def train(X, y, w, b, eta, epochs=1000):
    for _ in range(epochs):
        for x, y_true in zip(X, y):
            y_hat = predict(x, w, b)
            w = w + eta * (y_true - y_hat) * x
            b = b + eta * (y_true - y_hat)
    return w, b


## Inicialização dos dados e previsão inicial

Aqui definimos o conjunto de dados da porta lógica OR, inicializamos pesos e bias com valores arbitrários e calculamos as previsões iniciais do perceptron antes de qualquer treinamento. Em seguida, usamos um `DataFrame` do pandas para visualizar, lado a lado, as entradas `(X1, X2)`, os rótulos verdadeiros `y` e as previsões `y_hat`.


In [ ]:
# Dados de entrada (OR lógico)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([0, 1, 1, 1])

# Inicialização de pesos e bias
w = np.array([0.5, -1.5])
b = 2.0

# Previsão inicial (sem treinamento)
y_hat = np.zeros(len(X))
for i in range(len(X)):
    y_hat[i] = predict(X[i, :], w, b)

pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Treinamento do Perceptron

Aqui o perceptron é treinado utilizando os dados `X` e `y` com taxa de aprendizado `eta = 0.1`. Após o treinamento, imprimimos os novos valores de pesos `w` e bias `b` e mostramos, para cada amostra, o par `(y_true, y_hat)` para verificar se o modelo aprendeu a tarefa. Por fim, recalculamos `y_hat` com os parâmetros ajustados e exibimos um novo `DataFrame` para comparar os rótulos verdadeiros com as previsões corrigidas.


In [ ]:
# Reinicializa pesos e bias
w = np.array([0.5, -1.5])
b = 2.0

# Loop de treinamento
w, b = train(X, y, w, b, eta=0.1)
print("Parâmetros treinados:", w, b)

# Avaliação após o treinamento
y_hat = np.zeros(len(X))
for i in range(len(X)):
    y_hat[i] = predict(X[i, :], w, b)
    print(f"x: {X[i]}  y_true: {y[i]}  y_hat: {y_hat[i]}")

pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Versão aprimorada: classe `Perceptron`

Para tornar o código mais organizado e reutilizável, encapsulamos o perceptron em uma classe. A classe `Perceptron` inicializa pesos `w` e bias `b`, guarda a taxa de aprendizado `eta` e o número de épocas `epochs`, e fornece métodos para prever (`predict`) e treinar (`train`) o modelo usando a mesma regra de atualização vista anteriormente.


In [ ]:
class Perceptron:
    def __init__(self, input_size, eta=0.1, epochs=1000):
        self.w = np.zeros(input_size)
        self.b = 0.0
        self.eta = eta
        self.epochs = epochs

    def getParam(self):
        return {'w': self.w, 'b': self.b}

    def sigma(self, v):
        return 1 if v >= 0 else 0

    def predict(self, x):
        return self.sigma(self.w @ x + self.b)

    def train(self, X, y):
        w, b, eta = self.w, self.b, self.eta
        epochs = self.epochs

        for _ in range(epochs):
            for x, y_true in zip(X, y):
                y_hat = self.predict(x)
                w = w + eta * (y_true - y_hat) * x
                b = b + eta * (y_true - y_hat)

        self.w, self.b = w, b


In [ ]:
p = Perceptron(input_size=2, eta=0.1, epochs=1000)

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([0, 1, 1, 1])

p.train(X, y)
print("Parâmetros:", p.getParam())

y_hat = np.array([p.predict(x) for x in X])
pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Função sigmoide contínua e derivada

Nesta seção definimos a função sigmoide contínua, `sigma(z) = 1 / (1 + exp(-z))`, e sua derivada `diff_sigma(z) = s * (1 - s)`, onde `s = sigma(z)`. Avaliamos essas funções em um ponto específico e geramos valores em uma grade para poder plotar a curva da sigmoide e de sua derivada.


In [ ]:
def sigma(z):
    return 1.0 / (1.0 + np.exp(-z))


def diff_sigma(z):
    s = sigma(z)
    return s * (1 - s)

print("sigma(2.6) =", sigma(2.6))
print("diff_sigma(2.6) =", diff_sigma(2.6))

xx = np.linspace(-5, 5, 100)
zz = [sigma(x) for x in xx]
zz_diff = [diff_sigma(x) for x in xx]

plt.plot(xx, zz, label="sigmoide")
plt.plot(xx, zz_diff, label="derivada")
plt.legend()
plt.grid(True)
plt.show()


## Exemplo de rede neural multicamadas (MLP)

Agora estendemos a ideia do perceptron para uma rede neural com múltiplas camadas. Definimos tamanhos da camada de entrada, camada oculta e saída, inicializamos matrizes de pesos `W1` e `W2` com valores aleatórios em `[-1, 1]` e realizamos o **forward pass**: calculamos as ativações da camada oculta e depois da camada de saída, usando a função sigmoide. As saídas contínuas são arredondadas para obter previsões binárias e, por fim, exibimos um `DataFrame` comparando `y` e `y_hat`.


In [ ]:
from numpy.random import rand

np.random.seed(42)


def sigma(z):
    return 1.0 / (1.0 + np.exp(-z))


def diff_sigma(z):
    s = sigma(z)
    return s * (1 - s)

X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])
y = np.array([0, 1, 1, 1])

input_size = 2
hidden_size = 3
output_size = 1

# Transformação de [0,1] para [-1,1]
W1 = 2 * rand(input_size, hidden_size) - 1
W2 = 2 * rand(hidden_size, output_size) - 1

# Forward pass em lote
y_hat = np.zeros(len(X))

for i in range(len(X)):
    z1 = X[i] @ W1
    a1 = sigma(z1)

    z2 = a1 @ W2
    a2 = sigma(z2)

    y_hat[i] = np.round(a2)

pd.DataFrame({'X1': X[:, 0], 'X2': X[:, 1], 'y': y, 'y_hat': y_hat})


## Mapeamento linear entre intervalos com `encontra_coeficientes`

A função `encontra_coeficientes` encontra os coeficientes `a` e `b` de uma transformação linear `f(x) = a x + b` que leva um intervalo original `[x_min, x_max]` para um novo intervalo `[y_min, y_max]`. Mostramos exemplos de mapeamentos de `[0, 1]` para `[-1, 1]` e para `[-10, 10]`, além de aplicar a transformação `2 * X - 1` diretamente nos dados.


In [ ]:
def encontra_coeficientes(intervalo_original, intervalo_mapeado):
    A = np.array([[intervalo_original[0], 1],
                  [intervalo_original[1], 1]])
    b = np.array([[intervalo_mapeado[0]],
                  [intervalo_mapeado[1]]])
    return np.linalg.solve(A, b)

print("Mapeando [0,1] -> [-1,1]:", encontra_coeficientes([0, 1], [-1, 1]))
print("Mapeando [0,1] -> [-10,10]:", encontra_coeficientes([0, 1], [-10, 10]))

print("2*X - 1:
", 2 * X - 1)
